In [ ]:
import os
os.environ['GENERICIO_NO_MPI'] = 'true'
os.environ['PYVISTA_OFF_SCREEN'] = 'false'
os.environ['PYVISTA_USE_PANEL'] = '0'

import pyvista as pv
import pygio
import numpy as np

pv.set_jupyter_backend('trame')
pv.global_theme.jupyter_backend = 'trame'

# ==== CONFIGURATION ====
base_filename = "/projects/exasky/data/hacc/Argonne_L360_HACC001/STEP499/m000.full.mpicosmo.499"
num_files = 8  # Number of partition files

# ==== STEP 1: INSPECT FIRST FILE ====
print("Inspecting first file to detect available variables...")
first_file = f"{base_filename}#0"
sample_data = pygio.read_genericio(first_file)

available_vars = list(sample_data.keys())
print(f"Available variables: {available_vars}")
print(f"Total variables: {len(available_vars)}")

# Check what we have
has_mass = 'mass' in available_vars
has_uu = 'uu' in available_vars
has_rho = 'rho' in available_vars
has_phi = 'phi' in available_vars

print(f"\n✓ Position fields: x, y, z")
print(f"✓ Has 'mass': {has_mass}")
print(f"✓ Has 'uu' (internal energy): {has_uu}")
print(f"✓ Has 'rho' (density): {has_rho}")
print(f"✓ Has 'phi' (potential): {has_phi}")

# ==== STEP 2: LOAD ALL FILES ====
print(f"\nLoading all {num_files} files...")
all_positions = []
all_data = {var: [] for var in available_vars if var not in ['x', 'y', 'z']}

for i in range(num_files):
    filename = f"{base_filename}#{i}"
    print(f"Loading file {i+1}/{num_files}: {filename}")
    data = pygio.read_genericio(filename)
    
    # Positions (always present)
    positions = np.stack([data['x'], data['y'], data['z']], axis=1)
    all_positions.append(positions)
    
    # All other variables
    for var in all_data.keys():
        all_data[var].append(data[var])
    
    print(f"  ✓ {len(positions):,} particles")

# Concatenate
print("\nCombining all files...")
positions = np.vstack(all_positions)
for var in all_data.keys():
    all_data[var] = np.concatenate(all_data[var])

print(f"\nTOTAL PARTICLES: {len(positions):,}")
print(f"X range: [{positions[:, 0].min():.2f}, {positions[:, 0].max():.2f}]")
print(f"Y range: [{positions[:, 1].min():.2f}, {positions[:, 1].max():.2f}]")
print(f"Z range: [{positions[:, 2].min():.2f}, {positions[:, 2].max():.2f}]")

# ==== STEP 3: AUTO-DETECT BOUNDS ====
x_min, x_max = positions[:, 0].min(), positions[:, 0].max()
y_min, y_max = positions[:, 1].min(), positions[:, 1].max()
z_min, z_max = positions[:, 2].min(), positions[:, 2].max()

bounds = [[x_min, x_max], [y_min, y_max], [z_min, z_max]]
print(f"\nAuto-detected bounds: {bounds}")

# ==== STEP 4: SUBSAMPLE ====
subsample_factor = 30
indices = np.random.choice(len(positions), len(positions)//subsample_factor, replace=False)
points_subsample = positions[indices]
print(f"Subsampled to: {len(points_subsample):,} points")

# ==== STEP 5: CREATE GRIDS ====
grid_size = 128
print(f"\nCreating {grid_size}³ density grids...")

# Particle count density
hist_density, edges = np.histogramdd(
    positions, 
    bins=grid_size, 
    range=bounds
)

# Choose what to visualize based on available fields
grids_to_create = {}

if has_mass:
    print("  ✓ Creating mass-weighted density grid...")
    hist_mass, _ = np.histogramdd(
        positions, 
        bins=grid_size,
        range=bounds,
        weights=all_data['mass']
    )
    grids_to_create['mass'] = hist_mass

if has_uu:
    print("  ✓ Creating internal energy grid...")
    hist_uu, _ = np.histogramdd(
        positions,
        bins=grid_size,
        range=bounds,
        weights=all_data['uu']
    )
    hist_uu = np.divide(hist_uu, hist_density, where=hist_density>0)
    grids_to_create['uu'] = hist_uu

if has_rho:
    print("  ✓ Creating density (rho) grid...")
    hist_rho, _ = np.histogramdd(
        positions,
        bins=grid_size,
        range=bounds,
        weights=all_data['rho']
    )
    hist_rho = np.divide(hist_rho, hist_density, where=hist_density>0)
    grids_to_create['rho'] = hist_rho

if has_phi:
    print("  ✓ Creating potential (phi) grid...")
    hist_phi, _ = np.histogramdd(
        positions,
        bins=grid_size,
        range=bounds,
        weights=all_data['phi']
    )
    hist_phi = np.divide(hist_phi, hist_density, where=hist_density>0)
    grids_to_create['phi'] = hist_phi

# ==== SPATIAL CHECK ====
print("\n" + "="*50)
print("SPATIAL ALIGNMENT CHECK")
print("="*50)
print(f"\n📍 PARTICLE DATA:")
print(f"  Total particles: {len(positions):,}")
print(f"  X: [{x_min:.2f}, {x_max:.2f}]")
print(f"  Y: [{y_min:.2f}, {y_max:.2f}]")
print(f"  Z: [{z_min:.2f}, {z_max:.2f}]")

in_bounds = np.all(
    (positions >= [x_min, y_min, z_min]) &
    (positions <= [x_max, y_max, z_max]),
    axis=1
)
print(f"\n✓ Particles inside grid: {in_bounds.sum():,} / {len(positions):,} ({100*in_bounds.mean():.1f}%)")
print("="*50 + "\n")

# ==== VISUALIZATION ====
print("Creating PyVista visualization...")

pl = pv.Plotter(notebook=True)

# Add particle density volume (always available)
grid_density = pv.ImageData(dimensions=hist_density.shape)
grid_density.point_data['density'] = np.log10(hist_density.flatten(order='F') + 1)
grid_density.origin = (x_min, y_min, z_min)
grid_density.spacing = (
    (x_max - x_min) / grid_size,
    (y_max - y_min) / grid_size,
    (z_max - z_min) / grid_size
)

pl.add_volume(
    grid_density,
    scalars='density',
    cmap='viridis',
    opacity='sigmoid',
    name='Particle Count Density (log)'
)

# Add point cloud with available scalar fields
point_cloud = pv.PolyData(points_subsample)

# Add all available scalar fields to point cloud
scalar_field = None
for var in all_data.keys():
    point_cloud[var] = all_data[var][indices]
    if scalar_field is None:  # Use first available for coloring
        scalar_field = var

# Default to mass if available, otherwise first field
color_by = 'mass' if has_mass else (scalar_field if scalar_field else None)

if color_by:
    pl.add_mesh(
        point_cloud,
        scalars=color_by,
        cmap='plasma',
        point_size=2,
        render_points_as_spheres=True,
        opacity=0.5,
        name=f'Particles (colored by {color_by})'
    )
else:
    pl.add_mesh(
        point_cloud,
        color='white',
        point_size=2,
        render_points_as_spheres=True,
        opacity=0.5,
        name='Particles'
    )

# Camera setup
pl.camera_position = 'iso'
pl.add_axes()
pl.add_bounding_box()

print(f"\n✨ Visualization ready!")
print(f"   Coloring particles by: {color_by if color_by else 'default'}")
print(f"   Available fields: {list(all_data.keys())}")
pl.show(jupyter_backend='trame')